
# Used Car Price Prediction Using Artificial Neural Network (ANN)

## 1. Data Preprocessing

### Objective
This notebook prepares the used-car dataset for machine learning. It will standardize column names, assess data quality, handle missing or invalid values using documented rules, and export a separate cleaned dataset.

### Data Integrity
The original dataset in `data/raw/` will remain unchanged. All preprocessing outputs will be saved separately in `data/processed/`.


In [1]:
import sys
from pathlib import Path
import pandas as pd


# Make the project root accessible to Python.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import load_raw_data, standardize_column_names

# Load the raw dataset.
df_raw = load_raw_data()

print("Raw dataset loaded successfully.")
print("Original shape:", df_raw.shape)
print("\nOriginal column names:")
print(df_raw.columns.tolist())

# Standardize column names in a separate DataFrame.
df_standardized = standardize_column_names(df_raw)

print("\nStandardized column names:")
print(df_standardized.columns.tolist())
print("\nStandardized dataset shape:", df_standardized.shape)


Raw dataset loaded successfully.
Original shape: (2059, 20)

Original column names:
['Make', 'Model', 'Price', 'Year', 'Kilometer', 'Fuel Type', 'Transmission', 'Location', 'Color', 'Owner', 'Seller Type', 'Engine', 'Max Power', 'Max Torque', 'Drivetrain', 'Length', 'Width', 'Height', 'Seating Capacity', 'Fuel Tank Capacity']

Standardized column names:
['make', 'model', 'price', 'year', 'kilometer', 'fuel_type', 'transmission', 'location', 'color', 'owner', 'seller_type', 'engine', 'max_power', 'max_torque', 'drivetrain', 'length', 'width', 'height', 'seating_capacity', 'fuel_tank_capacity']

Standardized dataset shape: (2059, 20)


In [2]:
# Inspect data types after column standardization.
print("DATA TYPES")
display(
    df_standardized.dtypes.to_frame(name="data_type")
)

# Summarize missing values.
missing_summary = pd.DataFrame({
    "missing_count": df_standardized.isnull().sum(),
    "missing_percentage": (
        df_standardized.isnull().mean() * 100
    ).round(2),
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

print("\nMISSING VALUES")
if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary)

# Check the target variable.
print("\nTARGET VARIABLE: PRICE")
print("Missing prices:", df_standardized["price"].isnull().sum())
print("Minimum price:", df_standardized["price"].min())
print("Maximum price:", df_standardized["price"].max())

# Inspect sample values in specification columns.
specification_columns = [
    "engine",
    "max_power",
    "max_torque",
    "drivetrain",
    "length",
    "width",
    "height",
    "seating_capacity",
    "fuel_tank_capacity",
]

print("\nSPECIFICATION COLUMN SAMPLES")
for column in specification_columns:
    print(f"\n{column}:")
    display(df_standardized[column].dropna().head(5))


DATA TYPES


,data_type
make,str
model,str
price,int64
year,int64
kilometer,int64
fuel_type,str
transmission,str
location,str
color,str
owner,str



MISSING VALUES


,missing_count,missing_percentage
drivetrain,136,6.61
fuel_tank_capacity,113,5.49
engine,80,3.89
max_power,80,3.89
max_torque,80,3.89
length,64,3.11
width,64,3.11
height,64,3.11
seating_capacity,64,3.11



TARGET VARIABLE: PRICE
Missing prices: 0
Minimum price: 49000
Maximum price: 35000000

SPECIFICATION COLUMN SAMPLES

engine:


...

0    1505.0
1    1555.0
2    1550.0
3    1510.0
4    1795.0
Name: height, dtype: float64


seating_capacity:


0    5.0
1    5.0
2    5.0
3    5.0
4    7.0
Name: seating_capacity, dtype: float64


fuel_tank_capacity:


0    35.0
1    42.0
2    35.0
3    37.0
4    55.0
Name: fuel_tank_capacity, dtype: float64

In [3]:
from src.data_preprocessing import generate_data_quality_report, validate_required_columns

# Validate the required columns.
validate_required_columns(df_standardized)
print("Required-column validation passed.")

# Generate the complete data-quality report.
quality_report = generate_data_quality_report(df_standardized)

display(quality_report)

print("\nDuplicate rows:", df_standardized.duplicated().sum())


Required-column validation passed.


,data_type,missing_count,missing_percentage,unique_values
make,str,0,0.00,33
model,str,0,0.00,1050
price,int64,0,0.00,619
year,int64,0,0.00,22
kilometer,int64,0,0.00,847
fuel_type,str,0,0.00,9
transmission,str,0,0.00,2
location,str,0,0.00,77
color,str,0,0.00,17
owner,str,0,0.00,6



Duplicate rows: 0


In [4]:
from src.data_preprocessing import clean_text_values

df_text_cleaned = clean_text_values(df_standardized)

print("Text cleaning completed.")
print("Original shape:", df_standardized.shape)
print("Cleaned shape:", df_text_cleaned.shape)

print("\nOriginal data preview:")
display(df_standardized.head(3))

print("\nText-cleaned data preview:")
display(df_text_cleaned.head(3))

print(
    "\nBlank strings remaining in text columns:",
    sum(
        (
            df_text_cleaned[column]
            .astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
        ).sum()
        for column in df_text_cleaned.select_dtypes(
            include=["object", "string"]
        ).columns
    ),
)


Text cleaning completed.
Original shape: (2059, 20)
Cleaned shape: (2059, 20)

Original data preview:


,make,model,price,year,kilometer,fuel_type,transmission,location,color,owner,seller_type,engine,max_power,max_torque,drivetrain,length,width,height,seating_capacity,fuel_tank_capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0



Text-cleaned data preview:


,make,model,price,year,kilometer,fuel_type,transmission,location,color,owner,seller_type,engine,max_power,max_torque,drivetrain,length,width,height,seating_capacity,fuel_tank_capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0



Blank strings remaining in text columns: 0


In [5]:
from src.data_preprocessing import clean_numeric_columns

# Start with the text-cleaned dataset.
df_numeric_cleaned = clean_numeric_columns(
    df_text_cleaned,
    columns=["price", "year", "kilometer"],
)

print("Numeric conversion completed.")
print("Dataset shape:", df_numeric_cleaned.shape)

print("\nData types:")
display(
    df_numeric_cleaned[
        ["price", "year", "kilometer"]
    ].dtypes.to_frame(name="data_type")
)

print("\nMissing values after conversion:")
display(
    df_numeric_cleaned[
        ["price", "year", "kilometer"]
    ].isnull().sum().to_frame(name="missing_count")
)

print("\nNumeric summary:")
display(
    df_numeric_cleaned[
        ["price", "year", "kilometer"]
    ].describe()
)


Numeric conversion completed.
Dataset shape: (2059, 20)

Data types:


,data_type
price,int64
year,int64
kilometer,int64



Missing values after conversion:


,missing_count
price,0
year,0
kilometer,0



Numeric summary:


,price,year,kilometer
count,2.059000e+03,2059.000000,2.059000e+03
mean,1.702992e+06,2016.425449,5.422471e+04
std,2.419881e+06,3.363564,5.736172e+04
min,4.900000e+04,1988.000000,0.000000e+00
25%,4.849990e+05,2014.000000,2.900000e+04
50%,8.250000e+05,2017.000000,5.000000e+04
75%,1.925000e+06,2019.000000,7.200000e+04
max,3.500000e+07,2022.000000,2.000000e+06


In [6]:
from src.data_preprocessing import validate_numeric_values

numeric_validation = validate_numeric_values(df_numeric_cleaned)

print("NUMERIC DATA VALIDATION")
for check, count in numeric_validation.items():
    print(f"{check}: {count}")


NUMERIC DATA VALIDATION
missing_price: 0
non_positive_price: 0
missing_year: 0
year_before_1900: 0
year_after_current_year: 0
missing_kilometer: 0
negative_kilometer: 0


In [7]:
specification_columns = [
    "engine",
    "max_power",
    "max_torque",
    "drivetrain",
    "length",
    "width",
    "height",
    "seating_capacity",
    "fuel_tank_capacity",
]

for column in specification_columns:
    print(f"\n{'=' * 50}")
    print(f"Column: {column}")
    print("Data type:", df_text_cleaned[column].dtype)
    print("Missing values:", df_text_cleaned[column].isna().sum())
    print("Sample values:")
    display(
        df_text_cleaned[column]
        .dropna()
        .drop_duplicates()
        .head(10)
    )



Column: engine
Data type: string
Missing values: 80
Sample values:


0     1198 cc
1     1248 cc
2     1197 cc
4     2393 cc
5     1373 cc
6     1991 cc
7     1995 cc
8     1798 cc
9     1461 cc
11     999 cc
Name: engine, dtype: string


Column: max_power
Data type: string
Missing values: 80
Sample values:


0     87 bhp @ 6000 rpm
1     74 bhp @ 4000 rpm
2     79 bhp @ 6000 rpm
3     82 bhp @ 6000 rpm
4    148 bhp @ 3400 rpm
5     91 bhp @ 6000 rpm
6    181 bhp @ 5500 rpm
7    188 bhp @ 4000 rpm
8    177 bhp @ 5100 rpm
9     84 bhp @ 3750 rpm
Name: max_power, dtype: string


Column: max_torque
Data type: string
Missing values: 80
Sample values:


...

0    1505.0
1    1555.0
2    1550.0
3    1510.0
4    1795.0
5    1485.0
6    1432.0
7    1612.0
8    1476.0
9    1671.0
Name: height, dtype: float64


Column: seating_capacity
Data type: float64
Missing values: 64
Sample values:


0     5.0
4     7.0
24    4.0
37    8.0
43    2.0
45    6.0
Name: seating_capacity, dtype: float64


Column: fuel_tank_capacity
Data type: float64
Missing values: 113
Sample values:


0     35.0
1     42.0
3     37.0
4     55.0
5     43.0
7     51.0
8     50.0
10    45.0
11    28.0
18    32.0
Name: fuel_tank_capacity, dtype: float64

In [8]:
from src.data_preprocessing import extract_specification_values

df_specifications_cleaned = extract_specification_values(
    df_text_cleaned
)

specification_columns = [
    "engine",
    "max_power",
    "max_torque",
]

print("SPECIFICATION CONVERSION RESULTS")

for column in specification_columns:
    print(f"\n{column}")
    print(
        "Missing before:",
        df_text_cleaned[column].isna().sum(),
    )
    print(
        "Missing after:",
        df_specifications_cleaned[column].isna().sum(),
    )
    display(
        df_specifications_cleaned[column]
        .dropna()
        .head(5)
    )

print("\nDataset shape:", df_specifications_cleaned.shape)


SPECIFICATION CONVERSION RESULTS

engine
Missing before: 80
Missing after: 80


0    1198
1    1248
2    1197
3    1197
4    2393
Name: engine, dtype: Int64


max_power
Missing before: 80
Missing after: 80


0     87.0
1     74.0
2     79.0
3     82.0
4    148.0
Name: max_power, dtype: Float64


max_torque
Missing before: 80
Missing after: 80


0       109.0
1       190.0
2    112.7619
3       113.0
4       343.0
Name: max_torque, dtype: Float64


Dataset shape: (2059, 20)


In [9]:
from src.data_preprocessing import convert_measurement_columns

df_measurements_cleaned = convert_measurement_columns(
    df_specifications_cleaned
)

measurement_columns = [
    "length",
    "width",
    "height",
    "seating_capacity",
    "fuel_tank_capacity",
]

print("MEASUREMENT CONVERSION RESULTS")

display(
    df_measurements_cleaned[measurement_columns].dtypes.to_frame(
        name="data_type"
    )
)

print("\nMissing values after conversion:")
display(
    df_measurements_cleaned[measurement_columns]
    .isna()
    .sum()
    .to_frame(name="missing_count")
)

print("\nDataset shape:", df_measurements_cleaned.shape)


MEASUREMENT CONVERSION RESULTS


,data_type
length,float64
width,float64
height,float64
seating_capacity,float64
fuel_tank_capacity,float64



Missing values after conversion:


,missing_count
length,64
width,64
height,64
seating_capacity,64
fuel_tank_capacity,113



Dataset shape: (2059, 20)


In [10]:
# Use the latest cleaned DataFrame.
df_current = df_measurements_cleaned

missing_treatment_report = pd.DataFrame({
    "missing_count": df_current.isna().sum(),
    "missing_percentage": (
        df_current.isna().mean() * 100
    ).round(2),
    "data_type": df_current.dtypes.astype(str),
})

missing_treatment_report = (
    missing_treatment_report[
        missing_treatment_report["missing_count"] > 0
    ]
    .sort_values("missing_count", ascending=False)
)

print("MISSING-VALUE TREATMENT REPORT")
display(missing_treatment_report)

print("\nTotal rows:", len(df_current))
print("Total columns:", len(df_current.columns))

MISSING-VALUE TREATMENT REPORT


,missing_count,missing_percentage,data_type
drivetrain,136,6.61,string
fuel_tank_capacity,113,5.49,float64
engine,80,3.89,Int64
max_power,80,3.89,Float64
max_torque,80,3.89,Float64
length,64,3.11,float64
width,64,3.11,float64
height,64,3.11,float64
seating_capacity,64,3.11,float64



Total rows: 2059
Total columns: 20


In [11]:

from src.data_preprocessing import preprocess_dataset

# Reload the original dataset and apply all preprocessing steps.
df_raw = load_raw_data()
df_cleaned = preprocess_dataset(df_raw)

print("PREPROCESSING PIPELINE VALIDATION")
print("-" * 40)

print("Raw dataset shape:", df_raw.shape)
print("Cleaned dataset shape:", df_cleaned.shape)

print("\nColumn names:")
print(df_cleaned.columns.tolist())

print("\nFirst five cleaned records:")
display(df_cleaned.head())

print("\nData types:")
display(df_cleaned.dtypes.to_frame(name="data_type"))

print("\nMissing values by column:")
display(
    df_cleaned.isna()
    .sum()
    .to_frame(name="missing_count")
    .query("missing_count > 0")
)

print("\nDuplicate rows:", df_cleaned.duplicated().sum())

# Confirm that preprocessing did not alter the raw DataFrame.
print("\nRaw DataFrame remains unchanged:", df_raw.shape == (2059, 20))


PREPROCESSING PIPELINE VALIDATION
----------------------------------------
Raw dataset shape: (2059, 20)
Cleaned dataset shape: (2059, 20)

Column names:
['make', 'model', 'price', 'year', 'kilometer', 'fuel_type', 'transmission', 'location', 'color', 'owner', 'seller_type', 'engine', 'max_power', 'max_torque', 'drivetrain', 'length', 'width', 'height', 'seating_capacity', 'fuel_tank_capacity']

First five cleaned records:


,make,model,price,year,kilometer,fuel_type,transmission,location,color,owner,seller_type,engine,max_power,max_torque,drivetrain,length,width,height,seating_capacity,fuel_tank_capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198,87.0,109.0,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248,74.0,190.0,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197,79.0,112.7619,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197,82.0,113.0,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393,148.0,343.0,RWD,4735.0,1830.0,1795.0,7.0,55.0



Data types:


,data_type
make,string
model,string
price,int64
year,int64
kilometer,int64
fuel_type,string
transmission,string
location,string
color,string
owner,string



Missing values by column:


,missing_count
engine,80
max_power,80
max_torque,80
drivetrain,136
length,64
width,64
height,64
seating_capacity,64
fuel_tank_capacity,113



Duplicate rows: 0

Raw DataFrame remains unchanged: True


In [12]:
from src.data_preprocessing import save_cleaned_data, CLEANED_DATA_PATH

# Export the cleaned dataset.
saved_path = save_cleaned_data(df_cleaned)

# Reload the saved file to verify the export.
df_saved = pd.read_csv(saved_path)

print("CLEANED DATA EXPORT")
print("-" * 40)
print("Saved file:", saved_path)
print("File exists:", saved_path.exists())
print("Saved dataset shape:", df_saved.shape)
print("Column names:", df_saved.columns.tolist())
print("Duplicate rows:", df_saved.duplicated().sum())

print("\nMissing values in the exported CSV:")
display(
    df_saved.isna()
    .sum()
    .loc[lambda values: values > 0]
    .to_frame(name="missing_count")
)

assert saved_path.exists(), "Cleaned CSV was not created."
assert df_saved.shape == (2059, 20), "Unexpected dataset shape."
assert df_saved.columns.tolist() == df_cleaned.columns.tolist(), (
    "Exported column names do not match the cleaned DataFrame."
)

print("\nExport verification passed.")


CLEANED DATA EXPORT
----------------------------------------
Saved file: /Users/abhisheksharma/code/Projects/used_car_price_prediction/data/processed/car_details_cleaned.csv
File exists: True
Saved dataset shape: (2059, 20)
Column names: ['make', 'model', 'price', 'year', 'kilometer', 'fuel_type', 'transmission', 'location', 'color', 'owner', 'seller_type', 'engine', 'max_power', 'max_torque', 'drivetrain', 'length', 'width', 'height', 'seating_capacity', 'fuel_tank_capacity']
Duplicate rows: 0

Missing values in the exported CSV:


,missing_count
engine,80
max_power,80
max_torque,80
drivetrain,136
length,64
width,64
height,64
seating_capacity,64
fuel_tank_capacity,113



Export verification passed.


In [13]:
# Load the exported CSV independently.
df_final = pd.read_csv(CLEANED_DATA_PATH)

print("FINAL DATA QUALITY CHECK")
print("-" * 45)

# 1. Dataset dimensions
print("Rows:", len(df_final))
print("Columns:", len(df_final.columns))

# 2. Column naming convention
valid_column_names = all(
    column.replace("_", "").isalnum()
    and column == column.lower()
    for column in df_final.columns
)

print("Column naming convention valid:", valid_column_names)

# 3. Duplicate records
print("Duplicate rows:", df_final.duplicated().sum())

# 4. Required columns
required_columns = {
    "make", "model", "price", "year", "kilometer"
}
print(
    "Required columns present:",
    required_columns.issubset(df_final.columns),
)

# 5. Target integrity
print("Missing prices:", df_final["price"].isna().sum())
print("Non-positive prices:", (df_final["price"] <= 0).sum())

# 6. Missing values
print("\nMissing values by column:")
display(
    df_final.isna()
    .sum()
    .loc[lambda values: values > 0]
    .to_frame(name="missing_count")
)

# 7. Final assertions
assert df_final.shape == (2059, 20)
assert valid_column_names
assert required_columns.issubset(df_final.columns)
assert df_final["price"].notna().all()
assert (df_final["price"] > 0).all()
assert not df_final.duplicated().any()

print("\nAll final data-quality checks passed.")


FINAL DATA QUALITY CHECK
---------------------------------------------
Rows: 2059
Columns: 20
Column naming convention valid: True
Duplicate rows: 0
Required columns present: True
Missing prices: 0
Non-positive prices: 0

Missing values by column:


,missing_count
engine,80
max_power,80
max_torque,80
drivetrain,136
length,64
width,64
height,64
seating_capacity,64
fuel_tank_capacity,113



All final data-quality checks passed.



## 2. Preprocessing Summary and Conclusion

### Preprocessing Steps Completed
1. Standardized column names to lowercase snake_case.
2. Removed leading and trailing whitespace from text values.
3. Converted blank text values into missing values.
4. Converted price, year, and kilometres driven into numeric types.
5. Extracted numeric values from engine capacity, maximum power, and maximum torque specifications.
6. Ensured vehicle measurement columns use numeric data types.
7. Validated required columns, core numerical values, and duplicate records.
8. Exported the cleaned dataset to `data/processed/car_details_cleaned.csv`.

### Missing-Value Strategy
Missing vehicle specifications were retained rather than replaced with invented values. Missing-value imputation and categorical encoding will be performed within the model-training pipeline, fitted only on the training data to reduce data leakage.

### Final Dataset
- **Records:** 2,059
- **Columns:** 20
- **Target variable:** `price`
- **Target unit:** Indian Rupees (INR)
- **Duplicate records identified:** 0
- **Output file:** `data/processed/car_details_cleaned.csv`

### Conclusion
The dataset has been standardized and prepared for the next stage of the project. The original raw CSV remains unchanged, and the cleaned CSV provides a consistent input for model development.

The next stage will establish a baseline Linear Regression model and an Artificial Neural Network (ANN), followed by evaluation using MAE, RMSE, and R².
